# Brouillon — Détection de Fraude Assurance Automobile
**Type : Supervised Learning | Classification Binaire | Non Linéaire**

---
- Dataset : 1000 dossiers de sinistres
- Cible : `fraud_reported` → Y (fraude) / N (légitime)
- Déséquilibre : 75.3% légitimes vs 24.7% fraudes
- Modèles retenus : XGBoost + CatBoost

---
## ÉTAPE 1 — Chargement & Analyse Visuelle (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

# Chargement
df = pd.read_csv('../data/insurance_claims.csv')
df = df.replace('?', np.nan)

print('Shape brut:', df.shape)
print('\nRépartition cible:')
print(df['fraud_reported'].value_counts())
print(f'\nTaux de fraude: {(df["fraud_reported"]=="Y").mean():.1%}')

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Colonnes avec valeurs manquantes:')
print(missing)
print('\nColonne _c39 (parasite):', df['_c39'].isnull().sum(), 'NaN sur', len(df))

In [ ]:
# Visualisation 1 : Répartition fraude/légitime
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['fraud_reported'].value_counts()
axes[0].pie(counts, labels=['Légitime (N)', 'Fraude (Y)'],
            autopct='%1.1f%%', colors=['#66b3ff','#ff9999'],
            explode=(0, 0.1), startangle=90)
axes[0].set_title('Répartition : Fraude vs Légitime')

# Visualisation 2 : Montant réclamé par statut
sns.boxplot(data=df, x='fraud_reported', y='total_claim_amount',
            palette='Set2', ax=axes[1])
axes[1].set_title('Montant réclamé par statut de fraude')
axes[1].set_xlabel('Fraude (Y=1, N=0)')
axes[1].set_ylabel('Montant ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation 3 : Taux de fraude par hobby
h_fraud = df[df['fraud_reported'] == 'Y']['insured_hobbies'].value_counts()
h_total = df['insured_hobbies'].value_counts()
fraud_rate = (h_fraud / h_total).sort_values(ascending=False)

plt.figure(figsize=(14, 5))
sns.barplot(x=fraud_rate.index, y=fraud_rate.values, palette='magma')
plt.title('Taux de fraude par Hobby')
plt.ylabel('Taux de fraude')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation 4 : Matrice de corrélation
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.show()

---
## ÉTAPE 2 — Data Cleaning

In [ ]:
df_raw = df.copy()

# 1. Suppression des colonnes inutiles
cols_to_drop = ['policy_number', 'incident_location', 'insured_zip']
if '_c39' in df.columns:
    cols_to_drop.append('_c39')
df.drop(columns=cols_to_drop, inplace=True)
print(f'Colonnes supprimées : {cols_to_drop}')
print(f'Shape après suppression : {df.shape}')

In [ ]:
# 2. Conversion des dates
df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'], errors='coerce')
df['incident_date']    = pd.to_datetime(df['incident_date'], errors='coerce')
print('Dates converties en datetime')

In [ ]:
# 3. Standardisation des catégorielles (majuscules + strip)
cols_object = df.select_dtypes(include='object').columns.tolist()
for col in cols_object:
    df[col] = df[col].str.strip().str.upper()

print(f'Standardisation appliquée sur {len(cols_object)} colonnes')
print('\nValeurs incident_severity après standardisation:')
print(df['incident_severity'].value_counts())

In [ ]:
# 4. Vérification doublons
print(f'Doublons : {df.duplicated().sum()}')

# Résumé AVANT vs APRÈS
print('\n' + '='*50)
print('  RÉSUMÉ NETTOYAGE — AVANT vs APRÈS')
print('='*50)
print(f"  Lignes    : {df_raw.shape[0]:>6} → {df.shape[0]:>6}")
print(f"  Colonnes  : {df_raw.shape[1]:>6} → {df.shape[1]:>6}")
print(f"  NaN total : {df_raw.isnull().sum().sum():>6} → {df.isnull().sum().sum():>6}")

In [ ]:
# Export
df.to_csv('../process/insurance_claims_cleaned.csv', index=False)
print('Dataset nettoyé exporté : insurance_claims_cleaned.csv')
print(f'Taille finale : {df.shape[0]} lignes × {df.shape[1]} colonnes')

---
## ÉTAPE 3 — Modélisation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('../process/insurance_claims_cleaned.csv')
print('Dataset chargé:', df.shape)

In [ ]:
# Feature Engineering — 6 signaux experts
df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'], errors='coerce')
df['incident_date']    = pd.to_datetime(df['incident_date'], errors='coerce')
df['claim_delay_days'] = (df['incident_date'] - df['policy_bind_date']).dt.days

high_amount = df['total_claim_amount'].quantile(0.75)
low_premium = df['policy_annual_premium'].quantile(0.25)

df['rel1_early_incident']          = (df['claim_delay_days'] < 30).astype(int)
df['rel2_severity_amount_mismatch']= (df['incident_severity'].isin(['MINOR DAMAGE','TRIVIAL DAMAGE']) & (df['total_claim_amount'] > high_amount)).astype(int)
df['rel3_just_above_deductible']   = ((df['total_claim_amount'] > df['policy_deductable']) & (df['total_claim_amount'] < df['policy_deductable'] + 1000)).astype(int)
df['rel4_no_police_high_amount']   = ((df['police_report_available'] == 'NO') & (df['total_claim_amount'] > high_amount)).astype(int)
df['rel5_low_premium_high_claim']  = ((df['policy_annual_premium'] < low_premium) & (df['total_claim_amount'] > high_amount)).astype(int)
is_night = (df['incident_hour_of_the_day'] >= 23) | (df['incident_hour_of_the_day'] <= 5)
df['rel6_night_single_vehicle']    = (is_night & (df['incident_type'] == 'SINGLE VEHICLE COLLISION')).astype(int)
df['incident_month'] = df['incident_date'].dt.month
df['policy_year']    = df['policy_bind_date'].dt.year

print('Signaux experts créés:')
for col in ['rel1_early_incident','rel2_severity_amount_mismatch','rel3_just_above_deductible',
            'rel4_no_police_high_amount','rel5_low_premium_high_claim','rel6_night_single_vehicle']:
    print(f'  {col}: {df[col].sum()} cas positifs')

In [ ]:
# Préparation X / y + split
target = 'fraud_reported'
y = df[target].map({'N': 0, 'Y': 1})

selected_features = [
    'total_claim_amount', 'policy_deductable', 'policy_annual_premium',
    'incident_severity', 'police_report_available', 'incident_type',
    'witnesses', 'bodily_injuries', 'claim_delay_days',
    'incident_month', 'policy_year',
    'rel1_early_incident', 'rel2_severity_amount_mismatch',
    'rel3_just_above_deductible', 'rel4_no_police_high_amount',
    'rel5_low_premium_high_claim', 'rel6_night_single_vehicle',
    'age', 'months_as_customer', 'insured_hobbies'
]

X = df[selected_features]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Taux fraude train: {y_train.mean():.3f}')

In [ ]:
# Pipeline Preprocessing
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

print(f'Numériques: {len(num_cols)} | Catégorielles: {len(cat_cols)}')

In [ ]:
# Baseline — Logistic Regression
lr_pipe = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
lr_pipe.fit(X_train, y_train)
pred_lr   = lr_pipe.predict(X_test)
proba_lr  = lr_pipe.predict_proba(X_test)[:, 1]
print('=== Logistic Regression ===')
print(classification_report(y_test, pred_lr, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_lr), 3))

In [ ]:
# Baseline — Random Forest
rf_pipe = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(n_estimators=400, class_weight='balanced_subsample', random_state=42, n_jobs=-1))])
rf_pipe.fit(X_train, y_train)
pred_rf  = rf_pipe.predict(X_test)
proba_rf = rf_pipe.predict_proba(X_test)[:, 1]
print('=== Random Forest ===')
print(classification_report(y_test, pred_rf, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_rf), 3))

In [ ]:
# Baseline — Decision Tree
dt_pipe = Pipeline([('prep', preprocess), ('clf', DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42))])
dt_pipe.fit(X_train, y_train)
pred_dt  = dt_pipe.predict(X_test)
proba_dt = dt_pipe.predict_proba(X_test)[:, 1]
print('=== Decision Tree ===')
print(classification_report(y_test, pred_dt, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_dt), 3))

In [ ]:
# XGBoost — avec gestion du déséquilibre
xgb_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        use_label_encoder=False, eval_metric='logloss', random_state=42
    ))
])
xgb_pipe.fit(X_train, y_train)
pred_xgb  = xgb_pipe.predict(X_test)
proba_xgb = xgb_pipe.predict_proba(X_test)[:, 1]
print('=== XGBoost (baseline) ===')
print(classification_report(y_test, pred_xgb, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_xgb), 3))

In [ ]:
# CatBoost — avec gestion du déséquilibre
cat_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', CatBoostClassifier(verbose=0, auto_class_weights='Balanced', random_state=42))
])
cat_pipe.fit(X_train, y_train)
pred_cat  = cat_pipe.predict(X_test)
proba_cat = cat_pipe.predict_proba(X_test)[:, 1]
print('=== CatBoost (baseline) ===')
print(classification_report(y_test, pred_cat, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_cat), 3))

In [ ]:
# Tuning XGBoost — RandomizedSearchCV
xgb_tuned = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        use_label_encoder=False, eval_metric='logloss', random_state=42
    ))
])

param_grid_xgb = {
    'clf__n_estimators'    : [100, 500, 1000],
    'clf__max_depth'       : [3, 5, 7, 9],
    'clf__learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'clf__subsample'       : [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_tuned, param_distributions=param_grid_xgb,
    n_iter=10, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

print('Meilleurs params XGB:', xgb_search.best_params_)
pred_xgb_tuned  = best_xgb.predict(X_test)
proba_xgb_tuned = best_xgb.predict_proba(X_test)[:, 1]
print('\n=== XGBoost Tuné ===')
print(classification_report(y_test, pred_xgb_tuned, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_xgb_tuned), 3))

In [ ]:
# Tuning CatBoost — RandomizedSearchCV
cat_tuned = Pipeline([
    ('prep', preprocess),
    ('clf', CatBoostClassifier(verbose=0, auto_class_weights='Balanced', random_state=42))
])

param_grid_cat = {
    'clf__iterations'   : [100, 500, 1000],
    'clf__depth'        : [4, 6, 8],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__l2_leaf_reg'  : [1, 3, 5, 7]
}

cat_search = RandomizedSearchCV(
    cat_tuned, param_distributions=param_grid_cat,
    n_iter=5, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)
cat_search.fit(X_train, y_train)
best_cat = cat_search.best_estimator_

print('Meilleurs params CatBoost:', cat_search.best_params_)
pred_cat_tuned  = best_cat.predict(X_test)
proba_cat_tuned = best_cat.predict_proba(X_test)[:, 1]
print('\n=== CatBoost Tuné ===')
print(classification_report(y_test, pred_cat_tuned, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_cat_tuned), 3))

In [ ]:
# Optimisation du seuil de décision
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, proba_xgb_tuned)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f'Seuil par défaut : 0.50')
print(f'Seuil optimal (F1 max) : {best_threshold:.3f}')

pred_custom = (proba_xgb_tuned >= best_threshold).astype(int)
print('\n=== XGBoost Tuné (Seuil Optimisé) ===')
print(classification_report(y_test, pred_custom, digits=3))

In [ ]:
# Tableau comparatif final
results = {
    'Logistic Regression': roc_auc_score(y_test, proba_lr),
    'Random Forest'      : roc_auc_score(y_test, proba_rf),
    'Decision Tree'      : roc_auc_score(y_test, proba_dt),
    'XGBoost (tuné)'     : roc_auc_score(y_test, proba_xgb_tuned),
    'CatBoost (tuné)'    : roc_auc_score(y_test, proba_cat_tuned),
}

print('=== COMPARAISON ROC-AUC ===')
for model, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {model:<25} : {score:.3f}')

In [ ]:
# Sauvegarde des modèles
import joblib
joblib.dump(best_xgb, '../process/xgb_best_model.joblib')
joblib.dump(best_cat, '../process/cat_best_model.joblib')
print('Modèles sauvegardés !')